# 4.5 Attention 与 Transformer 组合

这一节进入更完整的神经网络模块：先把 Attention 的数据流拆清楚，再把多个 JIT kernel 组织成 Transformer block 的雏形。

前半部分关注 Attention：从 `Q @ K^T -> Softmax -> @ V` 到带 Q/K/V 投影和输出投影的完整 attention。后半部分关注函数组合：多个 JIT kernel 如何按顺序调用、复用，并最终拼成一个可验证的数据流。

## 1. 环境准备

本节会同时使用 matmul、transpose、reshape、softmax、LayerNorm、GELU 和 residual add，先统一准备运行环境。

In [ ]:
import os
from dataclasses import dataclass
from typing import Optional

import torch
import pypto
import numpy as np
from numpy.testing import assert_allclose

try:
    import torch_npu
except ImportError:
    torch_npu = None


def reset_pypto_notebook_state():
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        print("torch_npu is not available; the notebook will stay in SIM/CPU mode.")
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    torch.npu.set_device(device_id)
    return f"npu:{device_id}"


def current_device(device_id=None):
    if RUN_MODE == pypto.RunMode.NPU:
        if device_id is None:
            return device
        torch.npu.set_device(device_id)
        return f"npu:{device_id}"
    return "cpu"


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<not set>"))
print("device:", device)
print("run_mode:", RUN_MODE)
print("pypto:", pypto.__file__)


## 2. 学完后你应该能够

学完这一节后，可以回到这里检查自己是否已经做到：

1. 说清楚 scaled dot-product attention 的计算链。
2. 看懂 Q/K/V 投影、multi-head reshape/transpose 和输出投影的关系。
3. 理解多个 JIT kernel 顺序组合时 host 侧如何管理中间 Tensor。
4. 使用 residual connection、function reuse 和多函数组合搭出 Transformer block 雏形。
5. 看懂 core 函数复用和 JIT kernel 包装的关系。

## 3. 这一节会依次练什么

| 练习场景 | 位置 | 关键点 |
| --- | --- | --- |
| Scaled Dot-Product Attention | 4.1 | `Q @ K^T -> softmax -> @ V`。 |
| 带投影的 Attention | 4.2 | Q/K/V 投影、reshape、transpose、输出投影。 |
| 顺序函数组合 | 5.1 | LayerNorm 后接 GELU。 |
| 残差连接与函数复用 | 5.2 | 主干输出与旁路输入相加，同一 JIT kernel 用不同输入多次调用。 |
| Transformer Block 雏形 | 5.3 | 多个 kernel 组合成模块。 |
| Core 函数与 JIT Wrapper | 6.1 | 把计算表达式和 kernel 包装层拆开。 |

这一节和前面几节最大的不同是：重点不再只是“一个 kernel 怎么写”，而是“多个 kernel 和多个 Tensor 如何组织成模块”。

### 3.1 建议运行顺序

建议按“单个复杂算子 -> 完整模块 -> 多 kernel 组合”的顺序阅读和运行：

| 顺序 | 先看什么 | 重点问题 |
| --- | --- | --- |
| 1 | Attention 核心公式 | 每个中间张量的 shape 是什么？ |
| 2 | Q/K/V 投影与多头变换 | hidden states 如何变成 `[B, heads, S, D]`？ |
| 3 | 输出投影与写回 | 多头结果如何合并回 hidden size？ |
| 4 | 顺序函数组合 | host 侧如何分配并传递中间 Tensor？ |
| 5 | 残差连接与复用 | 哪些 kernel 可以独立复用，哪些 Tensor 必须 shape 对齐？ |
| 6 | Core + JIT Wrapper | 计算逻辑和参数/输出管理如何分层？ |

阅读时不需要急着记函数名，先沿着 Tensor 的流向走：输入从哪里来，中间结果放在哪里，最后和哪个 PyTorch reference 对齐。

## 4. Attention

Attention 的核心公式是：

```text
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

`Q @ K^T` 得到每个 query 对每个 key 的相关性分数，缩放后做 Softmax，最后用这些权重加权 Value。

### 阅读模板：先把 Attention 的 shape 看清楚

Attention 的核心张量通常是四维：

```text
q: [B, H, Q, D]
k: [B, H, K, D]
v: [B, H, K, D]
scores: [B, H, Q, K]
attn_weights: [B, H, Q, K]
context: [B, H, Q, D]
output: [B, H, Q, D]
```

其中 `B` 是 batch，`H` 是 head 数，`Q` 是 query 长度，`K` 是 key/value 长度，`D` 是每个 head 的维度。`transpose` 的作用是把 `K` 从 `[B, H, K, D]` 变成 `[B, H, D, K]`，这样 `Q @ K^T` 才能得到 `[B, H, Q, K]`。

In [ ]:
BATCH_SIZE = 2
SEQ_LEN_Q = 16
SEQ_LEN_KV = 16
SEQ_LEN = 32
NUM_HEADS = 8
HEAD_DIM = 64
HIDDEN_SIZE = 512


@dataclass
class AttentionConfig:
    num_heads: int = 8
    head_dim: int = 64
    scale: Optional[float] = None
    dtype: pypto.DataType = pypto.DT_BF16
    use_dynamic_shape: bool = False


def scaled_dot_product_attention_golden(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                                        scale: float, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
    scores = torch.matmul(q, k.transpose(-2, -1))
    scores = scores * scale
    if attn_mask is not None:
        scores = scores + attn_mask
    attn_weights = torch.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, v)


def scaled_dot_product_attention_core(q: pypto.Tensor, k: pypto.Tensor, v: pypto.Tensor,
                                      scale: float, dtype: pypto.DataType) -> pypto.Tensor:
    k_t = pypto.transpose(k, 2, 3)
    scores = pypto.matmul(q, k_t, out_dtype=dtype)
    scores_scaled = scores * scale
    attn_weights = pypto.softmax(scores_scaled, dim=-1)
    return pypto.matmul(attn_weights, v, out_dtype=dtype)


In [ ]:
@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def scaled_dot_product_attention_kernel(
    q: pypto.Tensor((BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM), pypto.DT_BF16),
    k: pypto.Tensor((BATCH_SIZE, NUM_HEADS, SEQ_LEN_KV, HEAD_DIM), pypto.DT_BF16),
    v: pypto.Tensor((BATCH_SIZE, NUM_HEADS, SEQ_LEN_KV, HEAD_DIM), pypto.DT_BF16),
    output: pypto.Tensor((BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM), pypto.DT_BF16)):
    scale = 1.0 / (HEAD_DIM ** 0.5)
    pypto.set_cube_tile_shapes([64, 64], [64, 64], [64, 64])
    pypto.set_vec_tile_shapes(1, 8, 16, HEAD_DIM)
    scores = pypto.matmul(q, pypto.transpose(k, 2, 3), out_dtype=pypto.DT_BF16)
    scores_scaled = pypto.mul(scores, scale)
    attn_weights = pypto.softmax(scores_scaled, dim=-1)
    output.move(pypto.matmul(attn_weights, v, out_dtype=pypto.DT_BF16))


def test_scaled_dot_product_attention(device_id=None, dynamic: bool = False) -> None:
    device_local = current_device(device_id)
    q_torch = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    k_torch = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN_KV, HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    v_torch = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN_KV, HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    out = torch.empty(BATCH_SIZE, NUM_HEADS, SEQ_LEN_Q, HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    scaled_dot_product_attention_kernel(q_torch, k_torch, v_torch, out)
    golden = scaled_dot_product_attention_golden(q_torch, k_torch, v_torch, 1.0 / (HEAD_DIM ** 0.5))
    print(f"Scaled attention input shape: {q_torch.shape}")
    print(f"Scaled attention output shape: {out.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        max_diff = (out - golden).abs().max().item()
        print(f"Scaled attention max diff: {max_diff:.6f}")
        assert max_diff < 3e-3, "Scaled attention mismatch!"


### 4.1 Scaled Dot-Product Attention

这个 kernel 是 Attention 的最小核心。它没有 Q/K/V 投影，也没有输出投影，只关注 attention 本身的三步：矩阵乘法、Softmax、矩阵乘法。

`pypto.transpose(k, 2, 3)` 对应 `K^T`，`pypto.softmax(scores_scaled, dim=-1)` 对应对每个 query 的 key 分数做归一化。

| 代码 | 输入 shape | 输出 shape | 含义 |
| --- | --- | --- | --- |
| `k_t = pypto.transpose(k, 2, 3)` | `[B, H, K, D]` | `[B, H, D, K]` | 把 key 的最后两维互换，准备与 query 做矩阵乘法。 |
| `scores = pypto.matmul(q, k_t, out_dtype=dtype)` | `[B, H, Q, D]` 和 `[B, H, D, K]` | `[B, H, Q, K]` | 计算 query-key 相似度。 |
| `scores_scaled = scores * scale` | `[B, H, Q, K]` | `[B, H, Q, K]` | 除以 `sqrt(d_k)`，避免分数过大。 |
| `attn_weights = pypto.softmax(scores_scaled, dim=-1)` | `[B, H, Q, K]` | `[B, H, Q, K]` | 沿 key 维归一化，得到权重。 |
| `return pypto.matmul(attn_weights, v, out_dtype=dtype)` | `[B, H, Q, K]` 和 `[B, H, K, D]` | `[B, H, Q, D]` | 用权重加权 value，得到 context。 |

这个核心版本最适合先理解 Attention 数学本身：先算相关性，再归一化，再聚合 Value。

In [ ]:
test_scaled_dot_product_attention()


In [ ]:
def attention_with_projection_golden(hidden_states: torch.Tensor, q_weight: torch.Tensor,
                                     k_weight: torch.Tensor, v_weight: torch.Tensor,
                                     out_weight: torch.Tensor) -> torch.Tensor:
    q = torch.matmul(hidden_states, q_weight)
    k = torch.matmul(hidden_states, k_weight)
    v = torch.matmul(hidden_states, v_weight)
    batch_size, seq_len, _ = q.shape
    q = q.view(batch_size, seq_len, NUM_HEADS, HEAD_DIM).transpose(1, 2)
    k = k.view(batch_size, seq_len, NUM_HEADS, HEAD_DIM).transpose(1, 2)
    v = v.view(batch_size, seq_len, NUM_HEADS, HEAD_DIM).transpose(1, 2)
    scores = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / (HEAD_DIM ** 0.5))
    attn_weights = torch.softmax(scores, dim=-1)
    context = torch.matmul(attn_weights, v)
    context = context.transpose(1, 2).reshape(batch_size, seq_len, NUM_HEADS * HEAD_DIM)
    return torch.matmul(context, out_weight)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def attention_with_projection_kernel(
    hidden_states: pypto.Tensor((BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE), pypto.DT_BF16),
    q_weight: pypto.Tensor((1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM), pypto.DT_BF16),
    k_weight: pypto.Tensor((1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM), pypto.DT_BF16),
    v_weight: pypto.Tensor((1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM), pypto.DT_BF16),
    out_weight: pypto.Tensor((1, NUM_HEADS * HEAD_DIM, HIDDEN_SIZE), pypto.DT_BF16),
    output_tensor: pypto.Tensor((BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE), pypto.DT_BF16)):
    tile_b = 1
    b_loop = BATCH_SIZE // tile_b
    scale = 1.0 / (HEAD_DIM ** 0.5)
    pypto.set_cube_tile_shapes([64, 64], [64, 64], [64, 64])
    pypto.set_vec_tile_shapes(1, 16, 8, HEAD_DIM)

    q_flat = pypto.matmul(hidden_states, q_weight, out_dtype=pypto.DT_BF16)
    k_flat = pypto.matmul(hidden_states, k_weight, out_dtype=pypto.DT_BF16)
    v_flat = pypto.matmul(hidden_states, v_weight, out_dtype=pypto.DT_BF16)
    q = pypto.transpose(pypto.reshape(q_flat, [BATCH_SIZE, SEQ_LEN, NUM_HEADS, HEAD_DIM]), 1, 2)
    k = pypto.transpose(pypto.reshape(k_flat, [BATCH_SIZE, SEQ_LEN, NUM_HEADS, HEAD_DIM]), 1, 2)
    v = pypto.transpose(pypto.reshape(v_flat, [BATCH_SIZE, SEQ_LEN, NUM_HEADS, HEAD_DIM]), 1, 2)

    for idx in pypto.loop(0, b_loop, 1, name="LOOP_L0_bIdx", idx_name="idx"):
        b_offset = idx * tile_b
        b_offset_end = ((idx + 1) * tile_b).min(BATCH_SIZE)
        view_shape = [tile_b, NUM_HEADS, SEQ_LEN, HEAD_DIM]
        valid_shape = [b_offset_end - b_offset, NUM_HEADS, SEQ_LEN, HEAD_DIM]
        q_view = pypto.view(q, view_shape, [b_offset, 0, 0, 0], valid_shape=valid_shape)
        k_view = pypto.view(k, view_shape, [b_offset, 0, 0, 0], valid_shape=valid_shape)
        v_view = pypto.view(v, view_shape, [b_offset, 0, 0, 0], valid_shape=valid_shape)
        scores = pypto.matmul(q_view, pypto.transpose(k_view, 2, 3), out_dtype=pypto.DT_BF16)
        scores_scaled = pypto.mul(scores, scale)
        attn_weights = pypto.softmax(scores_scaled, dim=-1)
        context = pypto.matmul(attn_weights, v_view, out_dtype=pypto.DT_BF16)
        context = pypto.transpose(context, 1, 2)
        context_flat = pypto.reshape(context, [tile_b, SEQ_LEN, NUM_HEADS * HEAD_DIM])
        output_view = pypto.matmul(context_flat, out_weight, out_dtype=pypto.DT_BF16)
        output_tensor[b_offset:b_offset_end, ...] = output_view


def test_attention_with_projection(device_id=None, dynamic: bool = False) -> None:
    device_local = current_device(device_id)
    hidden_states = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE, dtype=torch.bfloat16, device=device_local)
    q_weight = torch.randn(1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    k_weight = torch.randn(1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    v_weight = torch.randn(1, HIDDEN_SIZE, NUM_HEADS * HEAD_DIM, dtype=torch.bfloat16, device=device_local)
    out_weight = torch.randn(1, NUM_HEADS * HEAD_DIM, HIDDEN_SIZE, dtype=torch.bfloat16, device=device_local)
    out = torch.empty(BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE, dtype=torch.bfloat16, device=device_local)
    attention_with_projection_kernel(hidden_states, q_weight, k_weight, v_weight, out_weight, out)
    golden = attention_with_projection_golden(hidden_states, q_weight, k_weight, v_weight, out_weight)
    print(f"Attention projection hidden shape: {hidden_states.shape}")
    print(f"Attention projection output shape: {out.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        max_diff = (out - golden).abs().max().item()
        print(f"Attention projection max diff: {max_diff:.6f}")
        assert max_diff < 3e-3, "Attention projection mismatch!"


### 4.2 带投影的 Attention

完整 attention 比核心公式多了四个投影：`q_weight / k_weight / v_weight / out_weight`。前面三个把 hidden states 投影成 Q、K、V，最后一个把多头拼接后的结果投影回 hidden size。

这里最容易看混的是 shape：`[B, S, H] -> [B, S, heads * head_dim] -> [B, heads, S, head_dim]`。`reshape` 和 `transpose` 的作用就是让多个 head 能并行参与 attention。

| 代码 | 输入 shape | 输出 shape | 含义 |
| --- | --- | --- | --- |
| `q_flat = pypto.matmul(hidden_states, q_weight, ...)` | `[B, S, H]` 和 `[1, H, Hq]` | `[B, S, Hq]` | 先把 hidden states 线性投影成 Q。 |
| `k_flat / v_flat = pypto.matmul(...)` | `[B, S, H]` | `[B, S, Hq]` | 分别得到 K 和 V。 |
| `reshape(..., [B, S, NUM_HEADS, HEAD_DIM])` | `[B, S, Hq]` | `[B, S, heads, head_dim]` | 把最后一维拆成多头。 |
| `transpose(..., 1, 2)` | `[B, S, heads, head_dim]` | `[B, heads, S, head_dim]` | 把 head 维放到前面，便于后续 attention。 |
| `view(q, [tile, heads, S, D], ..., valid_shape=...)` | `[B, heads, S, D]` | `[tile, heads, S, D]` | 沿 batch 维取局部 tile。 |
| `context = pypto.matmul(attn_weights, v_view, ...)` | `[tile, heads, Q, K]` 和 `[tile, heads, K, D]` | `[tile, heads, Q, D]` | 用 attention 权重聚合 value。 |
| `transpose(context, 1, 2)` -> `reshape(...)` | `[tile, heads, Q, D]` | `[tile, Q, heads * D]` | 把多头结果合并回单个 hidden 维。 |
| `pypto.matmul(context_flat, out_weight, ...)` | `[tile, Q, heads * D]` 和 `[1, heads * D, H]` | `[tile, Q, H]` | 输出投影回 hidden size。 |
| `output_tensor[b_offset:b_offset_end, ...] = output_view` | 局部结果 | 全局输出 | 写回当前 batch tile。 |

这个例子把 attention 的“头拆分、头并行、头合并”流程都显式写出来了，所以非常适合作为 Transformer block 的前置准备。

In [ ]:
test_attention_with_projection()


## 5. 多函数组合模式

真实模块通常不会只靠一个 kernel 完成。这里练习的是另一种高级能力：把多个较小、职责清晰的 JIT kernel 组合起来，由 host 侧准备中间 Tensor 并顺序调用。

这种写法牺牲了一些单 kernel 的紧凑性，但换来了更清晰的模块边界和复用能力。

| 组合层次 | 代表含义 |
| --- | --- |
| 顺序组合 | 前一个 kernel 的输出作为后一个 kernel 的输入。 |
| 残差连接 | 原始输入绕过主干，与主干输出相加。 |
| 函数复用 | 同一个 JIT kernel 用不同输入多次调用。 |
| 模块拼装 | 多个小 kernel 组成 Transformer block。 |

这种风格和 PyTorch `nn.Module` 很接近，只是这里把模块边界显式暴露成了多个 JIT kernel。

In [ ]:
def layer_norm_golden(x: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor, eps: float) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    normalized = (x - mean) / torch.sqrt(var + eps)
    return normalized * gamma + beta


def gelu_golden(x: torch.Tensor) -> torch.Tensor:
    return torch.nn.functional.gelu(x)


def layernorm_core(x: pypto.Tensor, gamma: pypto.Tensor, beta: pypto.Tensor, eps: float = 1e-6) -> pypto.Tensor:
    hidden_size = x.shape[-1]
    mean = pypto.sum(x, dim=-1, keepdim=True) / hidden_size
    centered = x - mean
    var = pypto.sum(centered * centered, dim=-1, keepdim=True) / hidden_size
    normalized = centered / pypto.sqrt(var + eps)
    return normalized * gamma + beta


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def layer_norm_kernel(x: pypto.Tensor(), gamma: pypto.Tensor(), beta: pypto.Tensor(), out: pypto.Tensor()):
    pypto.set_vec_tile_shapes(64, 128)
    out[:] = layernorm_core(x, gamma, beta)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def linear_projection_kernel(x: pypto.Tensor(), weight: pypto.Tensor(), out: pypto.Tensor()):
    pypto.set_cube_tile_shapes([64, 64], [64, 64], [64, 64])
    out[:] = pypto.matmul(x, weight, out_dtype=x.dtype)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def gelu_activation_kernel(x: pypto.Tensor(), out: pypto.Tensor()):
    tile_shapes = [32 for _ in range(len(x.shape))]
    pypto.set_vec_tile_shapes(*tile_shapes)
    out[:] = x * pypto.sigmoid(x * 1.702)


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def residual_add_kernel(x: pypto.tensor(), residual: pypto.tensor(), out: pypto.tensor()):
    pypto.set_vec_tile_shapes(64, 128)
    out[:] = pypto.add(x, residual)


### 5.1 顺序函数组合

顺序函数组合的最小例子是：先 LayerNorm，再 GELU。两个 kernel 各自独立，host 侧负责准备 `normed` 和 `activated` 两个中间 Tensor。

| 步骤 | 输入 | 输出 | shape 变化 |
| --- | --- | --- | --- |
| LayerNorm | `x, gamma, beta` | `normed` | `[B, H] -> [B, H]` |
| GELU | `normed` | `activated` | `[B, H] -> [B, H]` |

这里的关键不是“两个 kernel 连起来就行”，而是 host 侧必须提前知道中间 Tensor 的 shape、dtype 和 device，并为它们分配好内存。

In [ ]:
def test_sequential_functions(device_id: int = None, dynamic: bool = False) -> None:
    device_local = current_device(device_id)
    batch_size, hidden_size = 32, 128
    x = torch.randn(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    gamma = torch.ones(hidden_size, dtype=torch.bfloat16, device=device_local)
    beta = torch.zeros(hidden_size, dtype=torch.bfloat16, device=device_local)
    normed = torch.empty(x.shape, dtype=torch.bfloat16, device=device_local)
    activated = torch.empty(x.shape, dtype=torch.bfloat16, device=device_local)
    layer_norm_kernel(x, gamma, beta, normed)
    gelu_activation_kernel(normed, activated)
    expected_normed = layer_norm_golden(x, gamma, beta, 1e-6)
    expected_activated = gelu_golden(expected_normed)
    print(f"Sequential functions input shape: {x.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert (normed - expected_normed).abs().max().item() < 1e-1
        assert (activated - expected_activated).abs().max().item() < 1e-1


def test_residual_connection(device_id: int = None, dynamic: bool = False) -> None:
    device_local = current_device(device_id)
    batch_size, hidden_size = 32, 128
    x = torch.randn(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    residual = torch.randn(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    out = torch.empty(x.shape, dtype=torch.bfloat16, device=device_local)
    residual_add_kernel(x, residual, out)
    expected = x + residual
    print(f"Residual connection shape: {x.shape}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert (out - expected).abs().max().item() < 1e-2


def test_function_reuse(device_id: int = None, dynamic: bool = True) -> None:
    device_local = current_device(device_id)
    batch_size, hidden_size = 32, 128
    gamma = torch.ones(hidden_size, dtype=torch.bfloat16, device=device_local)
    beta = torch.zeros(hidden_size, dtype=torch.bfloat16, device=device_local)
    inputs = [torch.randn(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local) for _ in range(3)]
    outputs = [torch.zeros(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local) for _ in range(3)]
    for x, out in zip(inputs, outputs):
        layer_norm_kernel(x, gamma, beta, out)
    print("Function reuse count: 3")
    if RUN_MODE == pypto.RunMode.NPU:
        for x, out in zip(inputs, outputs):
            expected = layer_norm_golden(x, gamma, beta, 1e-6)
            assert (out - expected).abs().max().item() < 1e-1


In [ ]:
test_sequential_functions()
test_residual_connection()
test_function_reuse()


### 5.2 残差连接与函数复用

残差连接和函数复用是工程中很常见的两个模式。

| 模式 | 代码表现 | 作用 |
| --- | --- | --- |
| residual connection | `residual_add_kernel(x, residual, out)` | 把输入或旁路结果加回主干，保持信息通路。 |
| function reuse | 多次调用 `layer_norm_kernel(x_i, gamma, beta, out_i)` | 同一个 kernel 用不同输入复用，减少重复定义。 |

残差连接要求两个输入 shape 能逐元素对齐；函数复用要求每次调用的参数 dtype、device 和 shape 约束符合 kernel 描述。

### 5.3 Transformer Block 雏形

Transformer block 雏形把多个 kernel 串起来：LayerNorm、线性投影、GELU、下投影和 residual add。这里更关注模块组织方式，所以中间的 `activated * up` 暂时用 PyTorch 计算，便于把注意力放在数据流和 kernel 编排上。

这说明工程里可以先用多个较小的 JIT kernel 建立正确的数据流，再逐步考虑是否把某些片段融合成更大的 kernel。

从 shape 角度看，Transformer block 的流程是：

```text
x -> layer_norm -> normed -> q/gate/up 投影 -> activation -> down 投影 -> ffn_out -> residual add -> output
```

这里用的是 LayerNorm + 两个 linear projection + GELU + residual。它不是完整 LLM block 的最终版，但已经把模块组装的骨架展示出来了。

In [ ]:
def test_transformer_block(device_id: int = None, dynamic: bool = False) -> None:
    device_local = current_device(device_id)
    batch_size, hidden_size, intermediate_size = 32, 128, 256
    x = torch.randn(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    gamma = torch.ones(hidden_size, dtype=torch.bfloat16, device=device_local)
    beta = torch.zeros(hidden_size, dtype=torch.bfloat16, device=device_local)
    gate_weight = torch.randn(hidden_size, intermediate_size, dtype=torch.bfloat16, device=device_local)
    up_weight = torch.randn(hidden_size, intermediate_size, dtype=torch.bfloat16, device=device_local)
    down_weight = torch.randn(intermediate_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    normed = torch.empty(x.shape, dtype=torch.bfloat16, device=device_local)
    gate = torch.zeros(batch_size, intermediate_size, dtype=torch.bfloat16, device=device_local)
    up = torch.zeros(batch_size, intermediate_size, dtype=torch.bfloat16, device=device_local)
    activated = torch.empty(gate.shape, dtype=torch.bfloat16, device=device_local)
    ffn_out = torch.zeros(batch_size, hidden_size, dtype=torch.bfloat16, device=device_local)
    output = torch.empty(x.shape, dtype=torch.bfloat16, device=device_local)

    layer_norm_kernel(x, gamma, beta, normed)
    linear_projection_kernel(normed, gate_weight, gate)
    linear_projection_kernel(normed, up_weight, up)
    gelu_activation_kernel(gate, activated)
    activated = activated * up
    linear_projection_kernel(activated, down_weight, ffn_out)
    residual_add_kernel(x, ffn_out, output)
    print(f"Transformer block input shape: {x.shape}")
    print(f"Transformer block output shape: {output.shape}")


In [ ]:
test_transformer_block()


## 6. Core 函数与 JIT Wrapper

这一部分展示另一种复用方式：先写一个普通 core 函数，再由 JIT kernel 调用它。这样可以把“计算表达式”从“kernel 参数和输出写回”中拆出来。

In [ ]:
SHAPE = (32, 32, 1, 256)
VAL = 1


def add_core(input0: pypto.Tensor, input1: pypto.Tensor, add1_flag: bool = False):
    pypto.set_vec_tile_shapes(1, 4, 1, 64)
    out = pypto.tensor(SHAPE, pypto.DT_FP32)
    if add1_flag:
        t3 = input0 + input1
        out[:] = t3 + VAL
    else:
        out[:] = input0 + input1
    return out


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def add_kernel(
    input0: pypto.Tensor(SHAPE, pypto.DT_FP32),
    input1: pypto.Tensor(SHAPE, pypto.DT_FP32),
    out: pypto.Tensor(SHAPE, pypto.DT_FP32),
    add1_flag: bool = True):
    out[:] = add_core(input0, input1, add1_flag)


def test_add_scalar_loop_multi_jit(device_id=None) -> None:
    device_local = current_device(device_id)
    input_data0 = torch.rand(SHAPE, dtype=torch.float32, device=device_local)
    input_data1 = torch.rand(SHAPE, dtype=torch.float32, device=device_local)
    golden = input_data0 + input_data1
    output_data = torch.empty(SHAPE, dtype=torch.float32, device=device_local)
    add_kernel(input_data0, input_data1, output_data, False)
    golden2 = input_data0 + input_data1 + VAL
    output_data2 = torch.empty(SHAPE, dtype=torch.float32, device=device_local)
    add_kernel(input_data0, input_data1, output_data2, True)
    print(f"multi_jit shape: {SHAPE}")
    if RUN_MODE == pypto.RunMode.NPU:
        assert_allclose(to_numpy(output_data), to_numpy(golden), rtol=3e-3, atol=3e-3)
        assert_allclose(to_numpy(output_data2), to_numpy(golden2), rtol=3e-3, atol=3e-3)


### 6.1 为什么要拆 core

拆出 core 函数之后，JIT kernel 只负责接收参数、调用 core、写回输出。对于较大的模块，这种写法能减少重复表达式，也让单元测试和阅读都更清楚。

| 层次 | 职责 |
| --- | --- |
| `add_core` | 写出具体计算逻辑。 |
| `add_kernel` | 作为 JIT 包装层，接收真实 Tensor 并把 core 的结果写回输出。 |
| 测试函数 | 构造输入、调用 kernel、写 PyTorch reference。 |

这种模式和函数式编程里的“纯函数 + 包装器”很像，只不过这里包装器是 PyPTO JIT kernel。

In [ ]:
test_add_scalar_loop_multi_jit()


## 7. 本节 API 速览

| API | 作用 |
| --- | --- |
| `pypto.matmul` | Attention 和线性投影的核心 |
| `pypto.transpose` | 构造 `K^T`，以及调整 multi-head 维度 |
| `pypto.reshape` | 在 `[B, S, H]` 与 `[B, S, heads, head_dim]` 之间转换 |
| `pypto.softmax` | 将 attention scores 归一化为权重 |
| `pypto.loop` | 按 batch tile 组织 attention |
| `pypto.view` | 取出当前 batch tile 的 Q/K/V |
| `pypto.set_cube_tile_shapes` | 配置矩阵乘法类计算 |
| `pypto.set_vec_tile_shapes` | 配置 softmax、GELU、residual 等向量计算 |

Attention 是前面所有基础能力的一次集中使用：matmul、transpose、reshape、softmax、tiling 和验证都同时出现。

| 组合 | 对应场景 |
| --- | --- |
| `matmul + transpose + softmax + matmul` | Scaled dot-product attention 核心。 |
| `matmul + reshape + transpose` | Q/K/V 投影后拆成多头。 |
| `transpose + reshape + matmul` | 多头结果合并后做输出投影。 |
| `JIT kernel -> host tensor -> JIT kernel` | 多函数顺序组合。 |
| `core function -> JIT wrapper` | 拆分计算逻辑和 kernel 包装层。 |

## 8. 本节自测

1. `Q @ K^T` 的输出 shape 是什么？
2. Attention 为什么要除以 `sqrt(head_dim)`？
3. Q/K/V 投影后为什么还要 reshape 和 transpose？
4. 多函数组合时，host 侧为什么要提前分配中间 Tensor？
5. `add_core` 和 `add_kernel` 分别负责什么？

参考答案：

1. 如果 `Q` 是 `[B, heads, Q, D]`，`K^T` 是 `[B, heads, D, K]`，输出就是 `[B, heads, Q, K]`。
2. 点积维度越大，score 数值越容易变大；除以 `sqrt(head_dim)` 可以让 softmax 前的分数更稳定。
3. 投影后还是扁平 hidden 维，需要 reshape 拆成多个 head，再 transpose 到 attention 需要的 `[B, heads, S, D]` 排布。
4. 因为每个 JIT kernel 只写自己的输出，跨 kernel 的数据流需要 host 侧 Tensor 承接。
5. `add_core` 负责表达计算公式，`add_kernel` 负责 JIT 包装、接收参数并写回输出。

这些问题能回答清楚，就说明已经能从“单算子”过渡到“模块组合”了。

## 9. 本节小结

到这里，你已经完成了从 Attention 核心公式到 Transformer block 组织方式的过渡。核心 attention 展示了复杂算子内部的数据流，带投影 attention 展示了完整模块的 shape 变化，多函数组合则展示了工程化组织方式。

下一节会进入系统与加速视角，继续学习 Cost Model 和 ACLGraph。